
# Ответ: для обычной: 0.927, для L2: 0.936

# 1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения-1 или 1.

In [1]:
import pandas as pd

data = pd.read_csv(r"/content/data-logistic.csv", header = None)
data.head()

X = data[[1, 2]]
y = data[0]

# 2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

In [2]:
import math

def e_distanse(q1, q2, p1, p2):
    return ((q1 - p1)**2 + (q2 - p2)**2)**0.5

def function(i, y, X, w1, w2, j):
    return y.iloc[i] * X.iloc[i, j] * (1 - 1 / (1 + math.exp(-y[i] * (w1 * X.iloc[i, 0] + w2 * X.iloc[i, 1]))))

# 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

In [3]:
l = len(X)
k = 0.1
e = 1e-5

def gradient(w1, w2):
    w1_new = w1 + k * (1 / l) * sum (function(i, y, X, w1, w2, 0) for i in range (l))
    w2_new = w2 + k * (1 / l) * sum (function(i, y, X, w1, w2, 1) for i in range (l))
    return (w1_new, w2_new)

def gradient_L2(w1_L2, w2_L2):
    w1_L2_new = w1_L2 + k * (1 / l) * sum (function(i, y, X, w1_L2, w2_L2, 0) for i in range (l)) - k * 10 * w1_L2
    w2_L2_new = w2_L2 + k * (1 / l) * sum (function(i, y, X, w1_L2, w2_L2, 1) for i in range (l)) - k * 10 * w2_L2
    return (w1_L2_new, w2_L2_new)

# 4.Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.

In [4]:
w1, w2 = 0, 0
iteration = 1
while iteration < 10000:
    w_vector = gradient(w1, w2)
    if e_distanse(w1, w2, w_vector[0], w_vector[1]) < e:
        w1, w2 = w_vector[0], w_vector[1]
        break
    else:
        w1, w2 = w_vector[0], w_vector[1]
        iteration += 1

w1_L2, w2_L2 = 0, 0
iteration_L2 = 1
while iteration_L2 < 10000:
    w_vector_L2 = gradient_L2(w1_L2, w2_L2)
    if e_distanse(w1_L2, w2_L2, w_vector_L2[0], w_vector_L2[1]) < e:
        w1_L2, w2_L2 = w_vector_L2[0], w_vector_L2[1]
        break
    else:
        w1_L2, w2_L2 = w_vector_L2[0], w_vector_L2[1]
        iteration_L2 += 1

# 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом. Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 +exp(−w1x1 −w2x2)).

In [5]:
from sklearn.metrics import roc_auc_score

y_binary = [1 if yi == 1 else 0 for yi in y]

p = [1/ (1+ math.exp(-(w1 * X.iloc[i, 0] + w2 * X.iloc[i, 1]))) for i in range (l)]
roc_auc = roc_auc_score(y_binary, p)

p_L2 = [1/ (1+ math.exp(-(w1_L2 * X.iloc[i, 0] + w2_L2 * X.iloc[i, 1]))) for i in range (l)]
roc_auc_L2 = roc_auc_score(y_binary, p_L2)

print(round(roc_auc, 3), round(roc_auc_L2, 3))

0.927 0.936


#6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги? Как меняется число итераций при уменьшении длины шага?

При $k = 0.1$ число итераций составляет 244. При увеличении шага до $k = 1.1$ число итераций падает до 29. Однако при слишком большом шаге ($k = 2$) алгоритм перестает сходиться. При уменьшении длины шага число итераций увеличивается.

# 7. Попробуйте менять начальное приближение. Влияет ли оно на что-нибудь?

Начальное приближение влияет только на число итераций, необходимых для достижения сходимости (например, при $w_1, w_2 = 10, 20$ итераций 723), но итоговое качество модели (AUC-ROC) остается практически неизменным.